# ClinicalBridge — Prototype (No-API)

**Educational prototype. All data is fictional. This system does not diagnose. Clinician review is
always required.**

This notebook replays *stored manual Claude outputs* (no API) through the real orchestration,
schema validation, and evaluation in `pipeline.py` + `evaluation/metrics.py`, then renders the
8-section Clinical Context Brief. See `docs/05_notebook_plan.md`.

## 1. Setup — import the shared pipeline

In [1]:
import sys, json, statistics
from pathlib import Path

def find_root(start=None):
    p = Path(start or Path.cwd()).resolve()
    for c in [p, *p.parents]:
        if (c / "data" / "patients.json").exists():
            return c
    raise FileNotFoundError("Run this notebook from inside the ClinicalBridge repo.")

ROOT = find_root()
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "evaluation"))
import pipeline as cb
import metrics
from IPython.display import Markdown, display

print("Project root:", cb.ROOT)
print(cb.DISCLAIMER)

Project root: C:\Users\karam\ClinicalBridge
EDUCATIONAL PROTOTYPE - all patient data is FICTIONAL. This is NOT a real clinical tool, it does NOT make diagnoses, and clinician review is ALWAYS required.


## 2. Load data + list scenarios

In [2]:
patients = cb.load_patients()
print(f"{len(patients)} fictional patients:", ", ".join(patients))
print("Scenarios:", {k: v.name for k, v in cb.SCENARIOS.items()})

10 fictional patients: P001, P002, P003, P004, P005, P006, P007, P008, P009, P010
Scenarios: {1: 'scenario_1_missed_medication.json', 2: 'scenario_2_false_alarm.json', 3: 'scenario_3_silent_deterioration.json', 4: 'scenario_4_incomplete_record.json', 5: 'scenario_5_conflicting_data.json'}


## 3. Run the pipeline and render the brief — Scenario 1 (Missed Medication)

`cb.run_pipeline` runs Triage -> EHR + Anamnesis -> Synthesis, validating every hand-off against a
JSON schema and enforcing the guard rails (patient-ID match, disclaimer, traceability, missing-data).

In [3]:
run1 = cb.run_pipeline(1, verbose=True)
display(Markdown(cb.render_brief(run1["brief"])))

Scenario 1 (Missed Medication): OK - schemas valid, guard rails passed [triage urgency=urgent (Urgent) validity=likely_valid] retrieval=langchain


# Clinical Context Brief  (BRIEF-P001-001)
_Patient P001 | Alert ALERT-P001-001 | 2026-06-20T08:05:00_

## 1. Alert Summary
Home BP 182/104 mmHg, well above the 160/100 alert limit and far above the patient's 126/76 baseline.

## 2. Patient Snapshot
- Age / Sex: 68 F
- Key conditions: Essential hypertension, Type 2 diabetes
- Key medications: Lisinopril 20 mg daily, Metformin 1000 mg BID, Atorvastatin 40 mg

## 3. Contextual Analysis
- BP has trended upward over the past week (150/92 -> 164/96 -> 176/100 -> 182/104), not a single isolated spike.  _[RPM.reading.001, RPM.reading.002, RPM.reading.003, RPM.alert]_
- Patient reports she ran out of her blood pressure pill about a week ago and has not refilled it.  _[ANAMNESIS.adherence.1]_
- Lisinopril 20 mg daily is an active antihypertensive; last refill 2026-05-02 was a 30-day supply, consistent with the medication running out around mid-June.  _[EHR.med.001]_
- Patient reports a mild headache over the past few days.  _[ANAMNESIS.symptom.1]_

## 4. Risk Assessment
- **Urgency (prioritization, NOT a diagnosis): Urgent [official taxonomy]** (internal: urgent) - Markedly elevated BP with an upward trend and a plausible, addressable cause; warrants prompt clinician review, though no reported red-flag symptoms.
- Factors to consider (NOT diagnoses):
    - **Possible contribution from an antihypertensive medication lapse** - Patient-reported that she ran out of her BP pill ~1 week ago; EHR shows lisinopril last refilled 2026-05-02 as a 30-day supply; BP has risen steadily over the same period.  _[ANAMNESIS.adherence.1, EHR.med.001, RPM.reading.001, RPM.alert]_
        - uncertainty: Self-reported lapse not yet confirmed against pharmacy fill data; other contributors (diet, stress, white-coat effect) not excluded.

## 5. Recommended Actions (for clinician review - NOT orders)
- Confirm the medication lapse with the patient and/or pharmacy and clarify the barrier to refill (reported lack of transportation).
- Consider a repeat BP measurement and ask about red-flag symptoms (severe headache, chest pain, visual changes, focal weakness).
- Consider whether recent renal function/electrolytes are needed before resuming/adjusting the ACE inhibitor.

## 6. Uncertainties and Gaps
- **Conflict:** The most recent clinic note (2026-04-15) records 'good adherence', whereas the patient now reports a lapse beginning ~1 week ago. Not necessarily contradictory (different time points), but the change since the last visit should be noted.  _[EHR.note.001, ANAMNESIS.adherence.1]_
- No serum potassium on file (last labs 2026-04-15 included creatinine/eGFR but not potassium) - relevant when adjusting an ACE inhibitor.
- Pharmacy fill/pickup history not available to confirm the reported lapse.
- No same-day repeat BP measurement or symptom check for end-organ concern.
- Overall confidence: **medium**

## 7. Sources Used
- `ANAMNESIS.adherence.1` -> ANAMNESIS
- `ANAMNESIS.symptom.1` -> ANAMNESIS
- `EHR.med.001` -> EHR
- `EHR.note.001` -> EHR
- `RPM.alert` -> RPM
- `RPM.reading.001` -> RPM
- `RPM.reading.002` -> RPM
- `RPM.reading.003` -> RPM

## 8. Safety Disclaimer
> This Clinical Context Brief is decision support only, generated from fictional data for an educational prototype. It does not provide a diagnosis or treatment recommendation. All clinical judgment remains with the reviewing clinician. Patient-reported items are self-reported and unverified.

## 4. Evaluate one scenario

In [4]:
def evaluate_scenario(sid):
    run = cb.run_pipeline(sid)
    sc = run["scenario"]
    return metrics.evaluate_run(run, sc, patients[sc["patient_id"]]["rpm"])

print(json.dumps(evaluate_scenario(1), indent=2))
metrics.print_human_checklist(cb.load_scenario(1)["gold_eval"])

{
  "triage": {
    "score": 1.0,
    "urgency_match": true,
    "validity_match": true,
    "urgency_direction": "match"
  },
  "retrieval": {
    "precision": 1.0,
    "recall": 1.0,
    "f1": 1.0,
    "missed": [],
    "extra": []
  },
  "anamnesis": {
    "score": 1.0,
    "recall": 1.0,
    "domain_coverage": 1.0,
    "missed": []
  },
  "hallucination_rate": 0.0,
  "traceability": 1.0,
  "hallucination_detail": {
    "hallucination_rate": 0.0,
    "traceability": 1.0,
    "total_claims": 6,
    "unsupported_claims": [],
    "index_coverage": 1.0
  },
  "safety_auto": {
    "disclaimer_present": true,
    "missing_data_section_present": true,
    "diagnosis_language_flags": [],
    "auto_pass": true
  }
}
Synthesis-accuracy checklist (human-scored):
  [ ] 1. Identifies medication-lapse as a factor to consider (not a diagnosis)
  [ ] 2. Connects EHR refill date/supply with the patient-reported lapse
  [ ] 3. Notes the rising BP trend rather than a single reading
  [ ] 4. Flags miss

## 5. Scorecard across all 5 scenarios

High scores here reflect that the data is **simulated and controlled** — see
`reports/evaluation_report.md` for the realistic interpretation and failure cases.

In [5]:
rows = []
for sid in sorted(cb.SCENARIOS):
    run = cb.run_pipeline(sid)
    sc = run["scenario"]
    ev = metrics.evaluate_run(run, sc, patients[sc["patient_id"]]["rpm"])
    rows.append({"scenario": f"{sid} {sc['name']}", "triage": ev["triage"]["score"],
                 "retr_f1": ev["retrieval"]["f1"], "anam": ev["anamnesis"]["score"],
                 "halluc": ev["hallucination_rate"], "trace": ev["traceability"],
                 "idx_cov": ev["hallucination_detail"]["index_coverage"],
                 "safety": "PASS" if ev["safety_auto"]["auto_pass"] else "CHECK"})

cols = ["scenario", "triage", "retr_f1", "anam", "halluc", "trace", "idx_cov", "safety"]
w = lambda h: 24 if h == "scenario" else 8
print("  ".join(h.ljust(w(h)) for h in cols))
print("-" * 92)
for r in rows:
    print("  ".join(str(r[c]).ljust(w(c)) for c in cols))
print("-" * 92)
for k in ["triage", "retr_f1", "anam", "halluc", "trace", "idx_cov"]:
    print(f"mean {k:8}: {round(statistics.mean(r[k] for r in rows), 3)}")

scenario                  triage    retr_f1   anam      halluc    trace     idx_cov   safety  
--------------------------------------------------------------------------------------------
1 Missed Medication       1.0       1.0       1.0       0.0       1.0       1.0       PASS    
2 False Alarm             1.0       1.0       1.0       0.0       1.0       1.0       PASS    
3 Silent Deterioration    1.0       1.0       1.0       0.0       1.0       1.0       PASS    
4 Incomplete Record       1.0       1.0       1.0       0.0       1.0       1.0       PASS    
5 Conflicting Data        1.0       1.0       1.0       0.0       1.0       1.0       PASS    
--------------------------------------------------------------------------------------------
mean triage  : 1.0
mean retr_f1 : 1.0
mean anam    : 1.0
mean halluc  : 0.0
mean trace   : 1.0
mean idx_cov : 1.0


## 6. Demo — Scenario 3 (Silent Deterioration): a trend no single reading reveals

In [6]:
run3 = cb.run_pipeline(3, verbose=True)
display(Markdown(cb.render_brief(run3["brief"])))
print(json.dumps(evaluate_scenario(3), indent=2))

Scenario 3 (Silent Deterioration): OK - schemas valid, guard rails passed [triage urgency=urgent (Urgent) validity=likely_valid] retrieval=langchain


# Clinical Context Brief  (BRIEF-P003-001)
_Patient P003 | Alert ALERT-P003-001 | 2026-06-20T07:25:00_

## 1. Alert Summary
Cumulative weight gain of +3.4 kg over 7 days (70.0 -> 73.4 kg) in a heart-failure monitoring program; device severity was rated 'medium' and no single 3-day window crossed the +2 kg limit.

## 2. Patient Snapshot
- Age / Sex: 76 F
- Key conditions: Chronic systolic heart failure, Atrial fibrillation, CKD stage 2, Hypertension
- Key medications: Furosemide 40 mg daily, Carvedilol 12.5 mg BID, Lisinopril 10 mg, Apixaban 5 mg BID

## 3. Contextual Analysis
- Weight rose steadily each day over 7 days (70.4 -> 70.9 -> 71.5 -> 72.0 -> 72.6 -> 73.1 -> 73.4 kg), a consistent upward trend rather than a one-day jump.  _[RPM.reading.001, RPM.reading.002, RPM.reading.003, RPM.reading.004, RPM.reading.005, RPM.reading.006, RPM.reading.007]_
- SpO2 has drifted down from a baseline of 96% to 94% (06-18) and 93% (06-20), still above the 92% alert limit.  _[RPM.baseline.spo2, RPM.reading.008, RPM.reading.009]_
- Patient reports increased shortness of breath on stairs and that her shoes/ankles feel tight and swollen.  _[ANAMNESIS.symptom.1, ANAMNESIS.symptom.2]_
- EHR documents chronic systolic heart failure with a dry weight ~70 kg and explicit instructions to call for weight rise >2 kg/3 days or increasing breathlessness.  _[EHR.dx.001, EHR.note.001]_
- On a daily diuretic (furosemide 40 mg); most recent BNP was elevated at 410 pg/mL (2026-03-01).  _[EHR.med.001, EHR.lab.001]_
- Patient reports two salty restaurant/takeout meals this week.  _[ANAMNESIS.lifestyle.1]_

## 4. Risk Assessment
- **Urgency (prioritization, NOT a diagnosis): Urgent [official taxonomy]** (internal: urgent) - Although no single reading breached a hard threshold and the device rated severity 'medium', the convergence of a sustained weight trend, SpO2 drift, and new symptoms in a heart-failure patient warrants prompt review. When signals disagree, the higher concern is retained.
- Factors to consider (NOT diagnoses):
    - **Possible worsening volume status / fluid retention to consider in the context of known heart failure** - Steady 7-day weight gain of +3.4 kg above dry weight, concurrent downward SpO2 drift, patient-reported exertional dyspnea and peripheral swelling, recent dietary salt, and a prior elevated BNP - all converging.  _[RPM.reading.007, RPM.baseline.weight, RPM.reading.009, ANAMNESIS.symptom.1, ANAMNESIS.symptom.2, ANAMNESIS.lifestyle.1, EHR.dx.001, EHR.lab.001]_
        - uncertainty: No clinical exam, no current weight confirmation on a calibrated scale, no current BNP/renal labs; the trend is suggestive but not diagnostic, and medication adherence is reported as good so this is not explained by a missed diuretic.

## 5. Recommended Actions (for clinician review - NOT orders)
- Consider prompt clinician review and direct contact with the patient despite the 'medium' device severity, given the converging trend and symptoms.
- Consider whether a clinical assessment of volume status (exam, current weight, symptom review) is warranted.
- Consider whether current renal function/electrolytes are needed before any diuretic adjustment.
- Reinforce the patient's instruction to report weight gain and breathlessness early (she reports not wanting to 'bother anyone').

## 6. Uncertainties and Gaps
- Most recent potassium and creatinine are from 2026-03-01 (>3 months old) - relevant given a loop diuretic and CKD.
- No current BNP to compare against the prior value of 410 pg/mL.
- SpO2 captured only twice over the window; no continuous trend.
- Home scale not confirmed to be calibrated/consistent.
- Overall confidence: **medium**

## 7. Sources Used
- `ANAMNESIS.lifestyle.1` -> ANAMNESIS
- `ANAMNESIS.symptom.1` -> ANAMNESIS
- `ANAMNESIS.symptom.2` -> ANAMNESIS
- `EHR.dx.001` -> EHR
- `EHR.lab.001` -> EHR
- `EHR.med.001` -> EHR
- `EHR.note.001` -> EHR
- `RPM.alert` -> RPM
- `RPM.baseline.spo2` -> RPM
- `RPM.baseline.weight` -> RPM
- `RPM.reading.001` -> RPM
- `RPM.reading.002` -> RPM
- `RPM.reading.003` -> RPM
- `RPM.reading.004` -> RPM
- `RPM.reading.005` -> RPM
- `RPM.reading.006` -> RPM
- `RPM.reading.007` -> RPM
- `RPM.reading.008` -> RPM
- `RPM.reading.009` -> RPM

## 8. Safety Disclaimer
> This Clinical Context Brief is decision support only, generated from fictional data for an educational prototype. It does not diagnose heart failure decompensation or any condition. All clinical judgment remains with the reviewing clinician. Patient-reported items are self-reported and unverified.

{
  "triage": {
    "score": 1.0,
    "urgency_match": true,
    "validity_match": true,
    "urgency_direction": "match"
  },
  "retrieval": {
    "precision": 1.0,
    "recall": 1.0,
    "f1": 1.0,
    "missed": [],
    "extra": []
  },
  "anamnesis": {
    "score": 1.0,
    "recall": 1.0,
    "domain_coverage": 1.0,
    "missed": []
  },
  "hallucination_rate": 0.0,
  "traceability": 1.0,
  "hallucination_detail": {
    "hallucination_rate": 0.0,
    "traceability": 1.0,
    "total_claims": 7,
    "unsupported_claims": [],
    "index_coverage": 1.0
  },
  "safety_auto": {
    "disclaimer_present": true,
    "missing_data_section_present": true,
    "diagnosis_language_flags": [],
    "auto_pass": true
  }
}


## 7. Genuine LangChain RAG retrieval (Documents + FAISS + local embeddings)

The EHR Retrieval Agent is grounded by a **real LangChain RAG pipeline** in `langchain_pipeline/`:
EHR records -> `Document` objects -> `RecursiveCharacterTextSplitter` -> local embeddings
(`sentence-transformers/all-MiniLM-L6-v2`, no API key) -> **FAISS** vector store -> top-k retriever.

If the LangChain stack is not installed the backend degrades to the local TF-IDF retriever
(`rag_retrieval.py`), so the notebook never breaks. Below we retrieve for Scenario 1 and score
**Precision@k / Recall@k** against the same `gold_relevant_ehr_ids` used everywhere else.

In [7]:
# The genuine LangChain RAG backend (falls back to local TF-IDF if LangChain is not installed).
backend = cb.get_retrieval_backend(prefer="auto")
print("Retrieval backend:", backend.name)
print("Config:", json.dumps(backend.config(), indent=2))

sid = 1
sc = cb.load_scenario(sid)
triage = cb.run_agent("triage", sid)
evidence = cb.retrieve_ehr_evidence(sid, k=5, backend=backend, triage=triage)
print("\nClinical question:\n ", evidence["question"])

print("\nRetrieved LangChain Documents (top-5):")
for rank, r in enumerate(evidence["results"], 1):
    score = r["score"]
    score_str = f"{score:.4f}" if isinstance(score, (int, float)) else "n/a"
    print(f"  {rank}. [{r['source_id']}] score={score_str} type={r['source_type']} patient={r['patient_id']}")
    print(f"       {r['text']}")

ranked = [r["source_id"] for r in evidence["results"]]
patk = metrics.score_retrieval_at_k(ranked, sc["gold_eval"], k=5)
print(f"\nPrecision@5 / Recall@5 vs gold relevant source IDs: "
      f"{patk['precision_at_k']} / {patk['recall_at_k']}  "
      f"(hits={patk['hits']}, missed={patk['missed'] or 'none'})")

Retrieval backend: langchain
Config: {
  "embedding_model": "sentence-transformers/all-MiniLM-L6-v2",
  "embedding_backend": "huggingface",
  "vector_store": "faiss",
  "num_documents": 71,
  "num_chunks": 71,
  "chunk_size": 512,
  "chunk_overlap": 64,
  "backend": "langchain"
}

Clinical question:
  blood pressure alert, value 182/104 mmHg. Related concepts: hypertension blood pressure antihypertensive renal kidney creatinine potassium egfr. Triage retrieval focus: antihypertensive medications with last refill date and days supply; renal function labs (creatinine, eGFR, potassium); hypertension and cardiac/renal diagnoses; recent BP-related visit notes. Retrieve the relevant diagnosis, medication, lab, and visit note for this patient.

Retrieved LangChain Documents (top-5):
  1. [EHR.dx.001] score=1.0798 type=diagnosis patient=P001
       Diagnosis: Essential hypertension (I10); status active; diagnosed 2016-03-12
  2. [EHR.lab.002] score=1.0987 type=lab patient=P001
       Lab: Crea